# Foundation models as feature generators

BONAFIDE can use machine learning models that live in **their own Python environment** as atom and bond feature generators. This requires two additional classes in order to connect a model:

- an **encoder** (runs in the model's environment): loads the model and returns its raw outputs for a molecule;
- a **featurizer** (runs here, in BONAFIDE): says where the model is, whether it takes 2D or 3D input, and how its outputs map onto BONAFIDE features.

BONAFIDE starts the encoder in a worker process, sends it each molecule, maps the outputs back onto its own atom and bond indices, and stores them like any other feature.

This notebook has two parts:

1. **A toy model** that runs with the BONAFIDE interpreter, to show the mechanics without any setup.
2. **A real model**: the 2D graph neural networks of [AcidAmine_Descriptor_Predict](https://github.com/nsf-c-cas/AcidAmine_Descriptor_Predict), which predict conformer-ensemble DFT descriptors of primary amines from SMILES. It needs its own environment and the model weights (see `acidamine/README.md`).

Run the notebook from the `examples/foundation_models` directory.

In [ ]:
import os
import sys

import pandas as pd

from bonafide import AtomBondFeaturizer
from bonafide.foundation import FoundationModelFeaturizer, Output

HERE = os.path.abspath("")
sys.path.insert(0, os.path.join(HERE, "acidamine"))  # for the AcidAmine example in part 2

# Delete the log file if it already exists:
if os.path.exists("bonafide.log"):
    os.remove("bonafide.log")


def print_last_error(log_file="bonafide.log"):
    # Print the last error from the BONAFIDE log file (errors are only written to the log)
    lines = open(log_file).read().splitlines()
    start = max(i for i, line in enumerate(lines) if "| ERROR |" in line)
    block = [lines[start]]
    for line in lines[start + 1 :]:
        if not line.startswith(" "):
            break
        block.append(line.strip())
    print("\n".join(block))

## Part 1: a toy model

### The encoder

The encoder runs in the model's environment, where BONAFIDE is (usually) not installed. It inherits from `FoundationModelEncoder`, which the worker makes importable as `bonafide_encoder`. It implements:

- `load(**options)`: called once when the worker starts;
- `encode(mol)`: called per molecule; returns a dictionary of named outputs (lists, numpy arrays or tensors).

`mol` carries the canonical SMILES string, a mol block, atomic numbers, bonds, coordinates (3D models only), charge and multiplicity.

The toy encoder computes Gasteiger charges, a pretend embedding, a lone-pair count for heteroatoms only, and bond orders:

In [ ]:
print(open("toy_encoder.py").read())

### The featurizer

The featurizer class is pure configuration. Each `Output` turns one encoder output into one BONAFIDE feature:

- `store="vector"` stores one list per atom as a comma-separated string (e.g., embeddings);
- `index=k` stores a single component of a vector output;
- `indices_key` names the output that says *which* atoms or bonds the values belong to (atom indices or atom pairs). Without it, an output must have one value per atom or bond.

`python` is the interpreter of the model's environment. The toy model runs with the current interpreter.

In [ ]:
class ToyModel(FoundationModelFeaturizer):
    origin = "toy"
    input_mode = "2D"  # the encoder gets the molecule as SMILES (no coordinates)
    encoder = os.path.join(HERE, "toy_encoder.py") + ":ToyEncoder"
    python = sys.executable
    options = {"embedding_scale": 1.0}  # passed to ToyEncoder.load()
    outputs = [
        Output("charge", "atom", name="gasteiger_charge"),
        Output("embedding", "atom", store="vector"),
        Output("embedding", "atom", name="degree", index=1),
        Output("lone_pairs", "atom", indices_key="lone_pair_atoms"),
        Output("bond_order", "bond", indices_key="bond_atoms"),
    ]

Registering the model adds one feature per `Output`. The returned dictionary maps the feature names (`{origin}{2D|3D}-{atom|bond}-{name}`) to their indices.

In [ ]:
f = AtomBondFeaturizer()
toy = f.add_foundation_model(ToyModel)
toy

In [ ]:
f.list_atom_features(origin="toy")

### Calculating the features

The first calculation starts the worker and loads the model; afterwards the worker stays alive.

In [ ]:
f.read_input(input_value="NCC(=O)Oc1ccccc1", namespace="ester")
f.show_molecule(index_type="atom")

In [ ]:
atom_features = [i for name, i in toy.items() if "-atom-" in name]
f.featurize_atoms(atom_indices="all", feature_indices=atom_features)
f.return_atom_features()

`lone_pairs` was only returned for the heteroatoms, so all other atoms are `_inaccessible`.

Bond features work the same way:

In [ ]:
f.featurize_bonds(bond_indices="all", feature_indices=toy["toy2D-bond-bond_order"])
f.return_bond_features()

### Atom order

The encoder works in the atom order of the canonical SMILES string; BONAFIDE maps the values back onto the atom order of the molecule as it was read in. Reading the same molecule in a different atom order gives the same values on the same atoms:

In [ ]:
f.read_input(input_value="c1ccc(cc1)OC(=O)CN", namespace="ester_reordered")
f.featurize_atoms(atom_indices="all", feature_indices=atom_features)
df = f.return_atom_features()
df.insert(0, "element", [a.GetSymbol() for a in f.mol_vault.mol_objects[0].GetAtoms()])
df

### Options and errors

Settings live under the model's origin and can be changed with `set_options()`. Changing the options starts a new worker with the new options:

In [ ]:
f.print_options(origin="toy")

In [ ]:
f.set_options(configs=("toy.options.embedding_scale", 10.0))
f.read_input(input_value="NCC(=O)Oc1ccccc1", namespace="ester")
f.featurize_atoms(atom_indices=[0, 1], feature_indices=toy["toy2D-atom-embedding"])
f.return_atom_features(atom_indices=[0, 1])

If the encoder raises an `EncoderError` (here: an element the toy model does not support), the features of that molecule fail with the error message, and BONAFIDE carries on with the next molecule. Errors are written to the log file, not raised:

In [ ]:
f.read_input(input_value="OB(O)c1ccccc1", namespace="boronic_acid")
f.featurize_atoms(atom_indices=0, feature_indices=toy["toy2D-atom-gasteiger_charge"])
print_last_error()

## Part 2: AcidAmine primary amine models

The [AcidAmine](https://github.com/nsf-c-cas/AcidAmine_Descriptor_Predict) models (*Digital Discovery*, 2025, **4**, 222) predict DFT descriptors of a primary amine's nitrogen (N1), its alpha carbon (C2) and the N1-C2 bond. Each descriptor comes as the Boltzmann-weighted (`boltz`), maximum, minimum and lowest-energy-conformer (`min_e`) value over the conformer ensemble, predicted from the SMILES string alone.

> These models were built for **primary amines** (one -NH2 bound to an sp3 carbon). Other molecules are rejected.

The encoder (`acidamine/amine_encoder.py`) and featurizer (`acidamine/amine_featurizer.py`) follow the same pattern as the toy model. Follow `acidamine/README.md` to set up the environment and download the weights, then set the three paths below. The interpreter can also come from the `BONAFIDE_ACIDAMINE_PYTHON` environment variable.

In [ ]:
# Basic setup, python paths and repo paths:
FOUNDATION_MODEL_PYTHON = os.environ.get(
    "BONAFIDE_ACIDAMINE_PYTHON", "../../.pixi/envs/acidamine-env/bin/python"
)
FOUNDATION_MDOEL_REPO = "../../../AcidAmine_Descriptor_Predict"

# Sometimes we will need to have the model files ready to load too:
FOUNDATION_MODEL_CHECKPOINTS = "../../../AcidAmine_Descriptor_Predict/2D/models/ckpts"

for path in [FOUNDATION_MODEL_PYTHON, FOUNDATION_MDOEL_REPO, FOUNDATION_MODEL_CHECKPOINTS]:
    print(os.path.exists(path), path)

In [ ]:
from pathlib import Path

from amine_featurizer import AcidAmineAmine2D

amine = f.add_foundation_model(AcidAmineAmine2D)
f.set_options(
    configs=[
        ("acidamine.python", str(Path(FOUNDATION_MODEL_PYTHON).resolve())),
        ("acidamine.options.repo_dir", str(Path(FOUNDATION_MDOEL_REPO).resolve())),
        ("acidamine.options.models_dir", str(Path(FOUNDATION_MODEL_CHECKPOINTS).resolve())),
    ]
)
amine_atom = [i for name, i in amine.items() if "-atom-" in name]
amine_bond = [i for name, i in amine.items() if "-bond-" in name]
print(len(amine_atom), "atom features,", len(amine_bond), "bond features")

The first calculation loads the 44 models (a few seconds); each further molecule takes about 0.3 s.

In [ ]:
f.read_input(input_value="NCC1CCCCC1", namespace="cyclohexylmethylamine")
f.show_molecule(index_type="atom")

In [ ]:
f.featurize_atoms(atom_indices="all", feature_indices=amine_atom)
f.featurize_bonds(bond_indices="all", feature_indices=amine_bond)

# Only N1 (index 0) and C2 (index 1) and the bond between them have values
f.return_atom_features(atom_indices=[0, 1]).T

In [ ]:
f.return_bond_features(bond_indices=0).T

### Comparing amines

The same featurizer can be used for one molecule after another; the worker keeps the models loaded. Here, the Boltzmann-weighted N1 descriptors for a few amines:

In [ ]:
from rdkit import Chem

amines = {
    "methylamine": "CN",
    "isopropylamine": "CC(C)N",
    "tert-butylamine": "CC(C)(C)N",
    "benzylamine": "NCc1ccccc1",
    "cyclohexylmethylamine": "NCC1CCCCC1",
}
descriptors = ["nbo_charge_boltz", "nbo_lp_energy_boltz", "v_bur_boltz", "pyrd_ar_boltz"]

rows = []
for name, smiles in amines.items():
    f.read_input(input_value=smiles, namespace=name)
    f.featurize_atoms(
        atom_indices="all", feature_indices=[amine[f"acidamine2D-atom-{d}"] for d in descriptors]
    )
    n1 = f.mol_vault.mol_objects[0].GetSubstructMatch(Chem.MolFromSmarts("[NH2]C"))[0]
    features = f.return_atom_features(atom_indices=n1)
    rows.append(
        {"amine": name, **{d: features[f"acidamine2D-atom-{d}"].iloc[0] for d in descriptors}}
    )

pd.DataFrame(rows).set_index("amine")

### Explicit hydrogen atoms and out-of-domain molecules

Explicit hydrogen atoms change BONAFIDE's atom indices; the values still land on N1 and C2:

In [ ]:
f.read_input(input_value="[H]N([H])C([H])([H])C([H])([H])[H]", namespace="ethylamine_with_H")
f.featurize_atoms(atom_indices="all", feature_indices=amine["acidamine2D-atom-nbo_charge_boltz"])
df = f.return_atom_features()
df.insert(0, "element", [a.GetSymbol() for a in f.mol_vault.mol_objects[0].GetAtoms()])
df

In [ ]:
# Purely illustrative, but for models where only certain molecules are valid
# we can build in fail-safes to prevent out-of-domain featurization from occurring:

# Aniline is not a primary alkyl amine: the encoder rejects it
f.read_input(input_value="Nc1ccccc1", namespace="aniline")
f.featurize_atoms(atom_indices=0, feature_indices=amine["acidamine2D-atom-nbo_charge_boltz"])
print_last_error()

### Cleaning up

The workers stop when the Python session ends. To free their resources earlier (e.g., GPU memory):

In [ ]:
f.close_foundation_models()